In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/pambifree/sms-dataset/sms_dataset_final.xlsx


In [2]:
import pandas as pd
from datasets import load_dataset

dfA = load_dataset("Nithingayakwad/Indian_Cyber_Scam_PhoneCall_Hinglish_Dataset")["train"].to_pandas()

dsB = load_dataset("sidzzz07/scamshield-dataset")
dfB = pd.concat([dsB[s].to_pandas().assign(orig_split=s) for s in dsB.keys()], ignore_index=True)

print("Dataset A (the 10k one):", dfA.shape)
print("Dataset B (ScamShield):", dfB.shape)
dfA.head()

README.md:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

India_Cyber_Scam_Hinglish_Dataset.csv:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/981 [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 37.5MB            

train.jsonl: downloading bytes:           |  0.00B            

... (3 outputs left out)

Generating validation split:   0%|          | 0/8192 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8202 [00:00<?, ? examples/s]

Dataset A (the 10k one): (10000, 8)
Dataset B (ScamShield): (85680, 10)


,text,label,scam_category,caller_type,audio_duration,urgency_level,contains_blackmail,language_style
0,"Hello ji Beta ghar aa gaya hoon, darwaza khol do.",0,none,family,34,low,False,hinglish
1,Namaste ma'am Project ka review meeting kal 4 ...,0,none,delivery_agent,54,low,False,hinglish
2,Ji namaskar Aapka KYC pending hai. Account 2 g...,1,relative,other_scammer,92,medium,False,hinglish
3,"Ji namaskar Doctor appointment kal 9 baje hai,...",0,none,delivery_agent,47,low,False,hinglish
4,Special Investigation Team se bol raha hoon. A...,1,bank_kyc,bank_scammer,74,medium,False,hinglish


In [3]:
import re

prefixes = ["Hello ji", "Hello sir", "Hello ma'am", "Namaste sir", "Namaste ma'am",
            "Ji namaskar", "Ji haan suniye", "Madam ek minute", "Sir ek minute suniye",
            "Aap sun rahe hain na?", "Dhyan se suniye"]
pattern = r"^(" + "|".join(re.escape(p) for p in prefixes) + r")\s+"
dfA["core"] = dfA["text"].str.replace(pattern, "", regex=True).str.strip()

print("Rows:", len(dfA))
print("Unique full texts:", dfA["text"].nunique())
print("Unique after removing greetings:", dfA["core"].nunique())
dfA["core"].value_counts().head(15)

Rows: 10000
Unique full texts: 743
Unique after removing greetings: 334


core
Papa train mein baith gaya hoon, safely pahunch jaunga.                              544
Bhai aaj shaam ko milte hain, coffee pe baat karte hain.                             526
Cab 3 minute mein pahunch raha hai.                                                  522
Aapka order deliver ho chuka hai. Feedback de dena.                                  506
Mummy main office pahunch gaya, aaj thoda late ho jaunga.                            500
School fees ki last date kal hai, payment kar dena.                                  496
Doctor appointment kal 9 baje hai, fasting rehna.                                    483
Beta ghar aa gaya hoon, darwaza khol do.                                             481
Sir kal subah 10 baje meeting hai, agenda mail kar diya.                             472
Project ka review meeting kal 4 baje hai kya?                                        470
Aapke father ko accident ho gaya hai. Turant 50,000 bhej dijiye hospital ke liye.    251
Aapka KYC pendin

In [4]:
parcel = dfA[dfA["core"].str.contains("clearance charge", case=False)]
print(parcel["scam_category"].value_counts())

scam_category
bank_kyc    154
amazon       67
aadhaar      51
lottery      51
relative     45
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

Xtr, Xte, ytr, yte = train_test_split(dfA["text"], dfA["label"], test_size=0.2,
                                      random_state=42, stratify=dfA["label"])

print("Test texts that also appear exactly in training:", round(Xte.isin(set(Xtr)).mean() * 100, 1), "%")

vec = TfidfVectorizer()
clf = LogisticRegression(max_iter=1000)
clf.fit(vec.fit_transform(Xtr), ytr)
print(classification_report(yte, clf.predict(vec.transform(Xte)), digits=4))

Test texts that also appear exactly in training: 99.4 %
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      1000
           1     1.0000    1.0000    1.0000      1000

    accuracy                         1.0000      2000
   macro avg     1.0000    1.0000    1.0000      2000
weighted avg     1.0000    1.0000    1.0000      2000



In [6]:
print(dfB["source_dataset"].value_counts(), "\n")
print(pd.crosstab(dfB["source_dataset"], dfB["is_scam"]), "\n")
print(pd.crosstab(dfB["language"], dfB["is_scam"]))

source_dataset
Smishing_Dataset                      69418
UCI_SMS_Spam                           5154
Kaggle_Hindi_Merged                    4567
Synthetic_Tier_C                       3720
Indian_Telecom_SMS                     2033
ysangam/Indian_Cyber_Scam_Hinglish      743
Benchmark_Ground_Truth                   45
Name: count, dtype: int64 

is_scam                                 0      1
source_dataset                                  
Benchmark_Ground_Truth                 32     13
Indian_Telecom_SMS                   1297    736
Kaggle_Hindi_Merged                  4474     93
Smishing_Dataset                    41457  27961
Synthetic_Tier_C                     2716   1004
UCI_SMS_Spam                         4512    642
ysangam/Indian_Cyber_Scam_Hinglish    110    633 

is_scam       0      1
language              
English   47295  29028
Hindi      5747    605
Hinglish   1556   1449


In [7]:
hing = dfB[dfB["language"] == "Hinglish"]
print(hing["source_dataset"].value_counts())

for src, g in hing.groupby("source_dataset"):
    print("\n=====", src, "=====")
    print("rows:", len(g), "| unique texts:", g["text"].nunique())
    for t in g["text"].sample(min(8, len(g)), random_state=1):
        print("-", t)

source_dataset
Synthetic_Tier_C                      1945
ysangam/Indian_Cyber_Scam_Hinglish     693
Smishing_Dataset                       348
Indian_Telecom_SMS                       9
Benchmark_Ground_Truth                   8
UCI_SMS_Spam                             2
Name: count, dtype: int64

===== Benchmark_Ground_Truth =====
rows: 8 | unique texts: 8
- Ramzan Mubarak! Pakistan Day Gala ends 26th March. Hurry up! shop FLAT 23% & 40% OFF before stock runs out. In-stores & online: https://bit.ly/3Yt8PDI T&Cs apply
To block promotional SMS from the J. Type "UNSUB" and send it to 2223623.

Hurry up! Only TWO days left to avail upto 50% OFF on all items at J. Summer Gala Sale, ending on 14th May. In-stores and online at: bit.ly/42P5HoZ T&Cs apply.
- Alert: Aapka bank account block hone wala hai kyunki aapne KYC update nahi kiya hai. Please apna details verify karein varna suvidhayen band kar di jayengi. Link: tinyurl.com/bank-fake-hindi
- Hi Beta
Silenced voice call
Unknown caller
Cy

In [8]:
eng = dfB[dfB["source_dataset"] == "Smishing_Dataset"]
for lab in [1, 0]:
    print(f"\n===== is_scam = {lab} =====")
    for t in eng[eng["is_scam"] == lab]["text"].sample(15, random_state=3):
        print("-", t[:150])


===== is_scam = 1 =====
- hi gopal   win an ipad by downloading the justdial app   httpjsdlinapps  zebrabarcode printer     sun data processing    ratings  okalipuram     kiran
- Dear Consumer Your Electricity power will be disconnect Tonight at 9:30 pm from electricity office because your previous month bill be not update plea
- your mobile number have won you rs   in ukindia chevrolet promo  to claim it send your name age phone no adress to chevroletawardplcgmailcom
- Dear Customer Your iPatym Store eK Y C have be expire today Please call us XXXXXXXXXX immediately Your Account will be block within 24hrs Thank You
- new excellent educational i is now using myly app to send all infoclick to download app httpgoogldnvltq  appusername  apppassword swat
- Dear Supr Customer you have Rs 100.00 Supr Credits remaining out of which Rs 100.00 Supr Credits will expire on XX-XX-XXXX Supr Credits can be use to 
- respected all madhusudan desi ghee rates date  kg tin consumer cow ceka cow jar it c

In [9]:
syn = dfB[dfB["source_dataset"] == "Synthetic_Tier_C"].copy()
syn["template"] = (syn["text"]
    .str.replace(r"https?://\S+|\b[\w-]+\.(?:com|net|live|xyz|info|in|org)\S*", "URL", regex=True)
    .str.replace(r"\d+", "0", regex=True))

print("Rows:", len(syn))
print("Unique texts:", syn["text"].nunique())
print("Unique after masking numbers and links:", syn["template"].nunique())
print("split_group values:", syn["split_group"].nunique())
syn["template"].value_counts().head(8)

Rows: 3720
Unique texts: 3720
Unique after masking numbers and links: 517
split_group values: 172


template
भारतीय रेल (IRCTC): पीएनआर 0 के लिए टिकट पुष्टि स्थिति: कंफर्म (Coach B0, Seat 0)। शुभ यात्रा!                                          315
Aapka Jio plan recharge of Rs 0,0 successful. 0% daily data balance valid till 0-Sep. Manage your plan on MyJio app.                    107
Aapka Jio plan recharge of Rs 0 successful. 0% daily data balance valid till 0-Sep. Manage your plan on MyJio app.                       80
Power Alert: Aapke ghar ki bijli aaj raat 0:0 disconnect kar di jayegi. Last month bill update nahi hua hai. Call Billing Officer: 0     48
IRCTC Alert: Train 0 departure rescheduled by 0 mins from Kolkata. PNR: 0. Check live status on NTES.                                    42
अमेज़न लॉगिन के लिए आपका एक बार का पासवर्ड (OTP) 0 है। यह 0 मिनट के लिए मान्य है। कृपया अपना ओटीपी किसी के साथ साझा न करें।              40
फ्लिपकार्ट लॉगिन के लिए आपका एक बार का पासवर्ड (OTP) 0 है। यह 0 मिनट के लिए मान्य है। कृपया अपना ओटीपी किसी के साथ साझा न करें।          40
स्विगी लॉगि

In [10]:
print("Rows with an exact duplicate elsewhere:", dfB["text"].duplicated(keep=False).sum())
multi = dfB.groupby("text")["source_dataset"].nunique()
print("Texts appearing in more than one source:", (multi > 1).sum())
conf = dfB.groupby("text")["is_scam"].nunique()
print("Texts with conflicting labels:", (conf > 1).sum())

for src in ["Indian_Telecom_SMS"]:
    print("\n=====", src, "=====")
    sample = dfB[dfB["source_dataset"] == src].sample(12, random_state=5)
    for t, y in zip(sample["text"], sample["is_scam"]):
        print(f"[{y}]", t[:140].replace("\n", " "))

print("\n===== Smishing_Dataset, Hinglish rows =====")
h = dfB[(dfB["source_dataset"] == "Smishing_Dataset") & (dfB["language"] == "Hinglish")].sample(12, random_state=5)
for t, y in zip(h["text"], h["is_scam"]):
    print(f"[{y}]", t[:140].replace("\n", " "))

Rows with an exact duplicate elsewhere: 0
Texts appearing in more than one source: 0
Texts with conflicting labels: 0

===== Indian_Telecom_SMS =====
[0] The above is what I want to share tonight. Regarding the candlestick I shared recently, I will organize it into a file tomorrow so that ever
[0] Oh yes I know
[0] It says account is blocked
[1] Here is your Exclusive chance to visit the movie sets of KALKI 2898 AD. Click here. open.wynk.in/kalki
[0] The week has ended with a small profit
[0] The master's predictions about the market in the past two weeks are completely correct. I believe the market may still fall... I will follow
[0] I won't share too much tonight, please enjoy this great festivalÃ°ÂÂÂ
[1] Hi Vi customer, A click makes your life so easy! View balance, make payment & do much more using Vi WhatsApp. Click wa.me/919654297000?text=
[1] LAST TWO DAYS Rs.250 credited as unconditional FREE UNLIMITED Cash Points in your account to shop at UNLIMITED Fashion Store.Hurry ! Tn

In [11]:
from sklearn.model_selection import GroupShuffleSplit

d = dfB[dfB["language"].isin(["English", "Hinglish"])].copy()
d = d[d["source_dataset"] != "Benchmark_Ground_Truth"]

# drop texts with conflicting labels, then exact duplicates
bad = conf[conf > 1].index
d = d[~d["text"].isin(bad)].drop_duplicates("text")

# cap the template-style ysangam rows at 3 per underlying sentence
is_ys = d["source_dataset"].str.contains("ysangam")
ys = d[is_ys].copy()
ys["core"] = ys["text"].str.replace(pattern, "", regex=True).str.strip()
ys = ys.groupby("core").head(3).drop(columns="core")
d = pd.concat([d[~is_ys], ys])

# sample English down (up to 4,000 per class); keep all Hinglish
eng = d[d["language"] == "English"]
hin = d[d["language"] == "Hinglish"]
eng = pd.concat([g.sample(min(len(g), 4000), random_state=42) for _, g in eng.groupby("is_scam")])
clean = pd.concat([eng, hin]).reset_index(drop=True)

# near-duplicate groups: first 6 words after masking numbers and links
masked = (clean["text"].str.lower()
          .str.replace(r"https?://\S+", "URL", regex=True)
          .str.replace(r"\d+", "0", regex=True))
clean["group"] = masked.str.split().str[:6].str.join(" ")

def split_by_group(df, test_size, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    a, b = next(gss.split(df, groups=df["group"]))
    return df.iloc[a], df.iloc[b]

trainval, test = split_by_group(clean, 0.15)
train, val = split_by_group(trainval, 0.15 / 0.85)

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"\n{name}: {len(part)} rows | unwanted share: {part['is_scam'].mean():.2f}")
    print(part["source_dataset"].value_counts().to_string())

print("\nGroup overlap train/test:", len(set(train["group"]) & set(test["group"])))
print("Group overlap train/val:", len(set(train["group"]) & set(val["group"])))


train: 7154 rows | unwanted share: 0.50
source_dataset
Smishing_Dataset                      5327
Synthetic_Tier_C                      1094
UCI_SMS_Spam                           338
ysangam/Indian_Cyber_Scam_Hinglish     244
Indian_Telecom_SMS                     151

val: 1485 rows | unwanted share: 0.55
source_dataset
Smishing_Dataset                      1161
Synthetic_Tier_C                       163
UCI_SMS_Spam                            85
ysangam/Indian_Cyber_Scam_Hinglish      44
Indian_Telecom_SMS                      32

test: 2066 rows | unwanted share: 0.40
source_dataset
Smishing_Dataset                      1168
Synthetic_Tier_C                       688
ysangam/Indian_Cyber_Scam_Hinglish     113
UCI_SMS_Spam                            66
Indian_Telecom_SMS                      31

Group overlap train/test: 0
Group overlap train/val: 0


In [12]:
vec = TfidfVectorizer()
clf = LogisticRegression(max_iter=1000)
clf.fit(vec.fit_transform(train["text"]), train["is_scam"])
print(classification_report(test["is_scam"], clf.predict(vec.transform(test["text"])), digits=4))

              precision    recall  f1-score   support

           0     0.9545    0.5584    0.7046      1241
           1     0.5910    0.9600    0.7316       825

    accuracy                         0.7188      2066
   macro avg     0.7728    0.7592    0.7181      2066
weighted avg     0.8094    0.7188    0.7154      2066



In [13]:
test = test.copy()
test["pred"] = clf.predict(vec.transform(test["text"]))
test["correct"] = test["pred"] == test["is_scam"]

print("Accuracy by source:")
print(test.groupby("source_dataset")["correct"].agg(["mean", "size"]).round(3))

fp = test[(test["is_scam"] == 0) & (test["pred"] == 1)]
print("\nFalse alarms (legit flagged as unwanted):", len(fp))
print(fp["source_dataset"].value_counts())
for t in fp.sample(12, random_state=1)["text"]:
    print("-", t[:130].replace("\n", " "))

Accuracy by source:
                                     mean  size
source_dataset                                 
Indian_Telecom_SMS                  0.903    31
Smishing_Dataset                    0.939  1168
Synthetic_Tier_C                    0.270   688
UCI_SMS_Spam                        0.924    66
ysangam/Indian_Cyber_Scam_Hinglish  1.000   113

False alarms (legit flagged as unwanted): 548
source_dataset
Synthetic_Tier_C      502
Smishing_Dataset       39
UCI_SMS_Spam            4
Indian_Telecom_SMS      3
Name: count, dtype: int64
- IRCTC Alert: Train 15431 departure rescheduled by 16 mins from Ahmedabad. PNR: 2039962746. Check live status on NTES.
- Aapka Jio plan recharge of Rs 450 successful. 100% daily data balance valid till 14-Sep. Manage your plan on MyJio app.
- Aapka Jio plan recharge of Rs 850 successful. 100% daily data balance valid till 16-Sep. Manage your plan on MyJio app.
- Aapka Jio plan recharge of Rs 7,200 successful. 100% daily data balance valid till 28-

In [14]:
print(fp["group"].nunique(), "distinct groups among", len(fp), "false alarms")

43 distinct groups among 548 false alarms


In [15]:
import pandas as pd

GOLD_PATH = "/kaggle/input/datasets/pambifree/sms-dataset/sms_dataset_final.xlsx"
g = pd.read_excel(GOLD_PATH)

print(g.shape)
print(list(g.columns))
g["label"] = (g["type"] != "legit").astype(int)        # 1 = unwanted (scam or promo), 0 = legit
g["text"] = (g["text"].str.replace("hxxp", "http", regex=False)
                      .str.replace("[dot]", ".", regex=False))
g["has_link"] = g["text"].str.contains(r"https?://|www\.|\.apk", case=False, regex=True)

rare_scam_tags = ["impersonation", "fee_scam", "reward_scam", "malware_link", "delivery_scam",
                  "job_scam", "utility_scam", "refund_scam", "giftcard_scam", "investment_scam", "otp_scam"]
g.loc[g["tag"].isin(rare_scam_tags), "tag"] = "other_scam"

assert g["origin"].isin(["human", "llm"]).all(), "origin must be 'human' or 'llm' on every row"

print(g["label"].value_counts(), "\n")
print(g["origin"].value_counts(), "\n")
print(g.groupby("type")["has_link"].sum())
g.to_csv("/kaggle/working/gold_test_v1.csv", index=False)

(326, 5)
['text', 'label', 'type', 'tag', 'origin']
label
1    183
0    143
Name: count, dtype: int64 

origin
llm      309
human     17
Name: count, dtype: int64 

type
legit    20
promo    15
scam     55
Name: has_link, dtype: int64


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(pd.concat([train["text"], g["text"]]).str.lower())
sim = cosine_similarity(v.transform(g["text"].str.lower()), v.transform(train["text"].str.lower()))
g["max_sim_to_train"] = sim.max(axis=1)

print((g["max_sim_to_train"] > 0.8).sum(), "gold messages have a near-duplicate in train")
g.sort_values("max_sim_to_train", ascending=False).head(10)[["text", "max_sim_to_train", "origin"]]

0 gold messages have a near-duplicate in train


,text,max_sim_to_train,origin
8,Your electricity will be disconnected tonight....,0.761379,human
157,Covid-19 Vaccination certificate is available....,0.575235,llm
230,Are you coming to the office today? We have a ...,0.555003,llm
105,Your lenses need an upgrade. Get 2 pairs of ey...,0.505438,llm
78,Your Netflix subscription payment failed. Your...,0.485834,llm
234,Happy Birthday! Hope you have a wonderful year...,0.469462,llm
1,"Congratulations! You have won ₹4,999 cashback....",0.458251,human
80,Your Apple ID is locked due to suspicious acti...,0.451371,llm
187,Thank you for shopping at Shoppers Stop. Your ...,0.445096,llm
51,Kijiye apna KYC update turant nahi toh a/c blo...,0.439204,llm


In [17]:
g.to_csv("/kaggle/working/gold_test_v1.csv", index=False)

In [18]:
import os
os.makedirs("/kaggle/working/public_mix_v1", exist_ok=True)
for name, part in [("train", train), ("val", val), ("test", test)]:
    part[["text", "is_scam", "source_dataset", "language", "group"]].to_csv(
        f"/kaggle/working/public_mix_v1/{name}.csv", index=False)